## Task 4.4: FedSAM - Sharpness-Aware Minimization in FL
**Goal: Find flatter minima to improve generalization under heterogeneity**

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from scriptsfl.data_utils import load_dataset, create_dirichlet_split, get_dataloaders
from scriptsfl.models import get_model
from scriptsfl.federated_utils import get_model_weights, set_model_weights, evaluate_model
from scriptsfl.server import Server
from scriptsfl.results_utils import save_results, plot_comparison

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
print("="*80)
print("TASK 4.4: FEDSAM IMPLEMENTATION")
print("="*80)

#%% Configuration
CONFIG = {
    'dataset': 'cifar10',
    'num_clients': 5,
    'num_rounds': 50,
    'local_epochs': 5,
    'batch_size': 32,
    'lr': 0.01,
    'momentum': 0.9,
    'rho': 0.05,  # SAM perturbation radius
    'alpha': 0.1,  # Data heterogeneity
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

#%% Load data (reuse scripts)
train_dataset, test_dataset = load_dataset(CONFIG['dataset'])
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

client_indices = create_dirichlet_split(train_dataset, CONFIG['num_clients'], alpha=CONFIG['alpha'])
client_loaders = get_dataloaders(train_dataset, client_indices, batch_size=CONFIG['batch_size'])

## SAM Training Function + classes

In [ ]:
def sam_step(model, data, target, criterion, optimizer, rho=0.05):
    """
    Perform one SAM optimization step

    SAM Algorithm:
    1. Compute gradients at current weights w
    2. Find adversarial perturbation: w_adv = w + rho * grad / ||grad||
    3. Compute gradients at w_adv
    4. Update weights using gradients from w_adv

    Args:
        model: PyTorch model
        data: input batch
        target: target labels
        criterion: loss function
        optimizer: optimizer
        rho: perturbation radius

    Returns:
        loss value
    """
    # TODO: STEP 1 - First forward/backward to compute gradients
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()

    # TODO: STEP 2 - Compute adversarial perturbation
    # For each parameter, compute: epsilon = rho * grad / ||grad||
    with torch.no_grad():
        # TODO: Store original parameters
        # TODO: Compute gradient norm
        # TODO: Apply perturbation: param += epsilon
        pass

    # TODO: STEP 3 - Second forward/backward at perturbed weights
    optimizer.zero_grad()
    # TODO: Forward pass at perturbed weights
    # TODO: Compute loss and gradients

    # TODO: STEP 4 - Restore original parameters and apply update
    with torch.no_grad():
        # TODO: Restore parameters to original values
        pass

    # TODO: Apply optimizer step (using gradients from perturbed point)
    optimizer.step()

    return loss.item()

In [ ]:
#%% FedSAM Client
class FedSAMClient:
    """
    Client using SAM for local training
    """

    def __init__(self, client_id, data_loader, model, device='cpu'):
        self.client_id = client_id
        self.data_loader = data_loader
        self.device = device
        self.model = copy.deepcopy(model).to(device)
        self.data_size = len(data_loader.dataset)

    def train_local(self, epochs, lr, rho, momentum=0.9):
        """
        Train locally using SAM
        """
        self.model.train()
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(self.model.parameters(), lr=lr, momentum=momentum)

        epoch_losses = []

        for epoch in range(epochs):
            running_loss = 0.0
            for data, target in self.data_loader:
                data, target = data.to(self.device), target.to(self.device)

                # TODO: Use sam_step instead of regular SGD step
                loss = sam_step(self.model, data, target, criterion, optimizer, rho)

                running_loss += loss * data.size(0)

            epoch_losses.append(running_loss / self.data_size)

        return {'client_id': self.client_id, 'final_loss': epoch_losses[-1]}

    def get_weights(self):
        return get_model_weights(self.model)

    def set_weights(self, weights):
        set_model_weights(self.model, weights)

    def get_data_size(self):
        return self.data_size

In [ ]:
#%% FedSAM Server (uses standard aggregation)

class FedSAMServer(Server):
    """
    Server for FedSAM (uses FedSAM clients)
    """

    def train_round(self, local_epochs, lr, rho, momentum=0.9):
        """
        One round of FedSAM training
        """
        # Select clients
        selected_clients = self.select_clients(fraction=1.0)

        # Broadcast
        self.broadcast_weights(selected_clients)

        # Local training with SAM
        for client in selected_clients:
            client.train_local(epochs=local_epochs, lr=lr, rho=rho, momentum=momentum)

        # Aggregate (standard FedAvg aggregation)
        self.aggregate(selected_clients)

        # Evaluate
        test_acc, test_loss = self.evaluate()

        return {'test_accuracy': test_acc, 'test_loss': test_loss}

## Run Experiments


In [ ]:

print("\n--- Comparing FedAvg vs FedSAM ---")

# TODO: Run FedAvg baseline (can load from Task 4.1)

# TODO: Run FedSAM with different rho values
rho_values = [0.0, 0.01, 0.05, 0.1]  # 0.0 = FedAvg
results_fedsam = {}

for rho in rho_values:
    method_name = 'FedAvg' if rho == 0.0 else f'FedSAM (ρ={rho})'
    print(f"\n--- Training with {method_name} ---")

    # TODO: Initialize and train
    # ...

    pass

# TODO: Compare and plot results

In [ ]:
print("\n" + "="*80)
print("TASK 4.4 IMPLEMENTATION NOTES")
print("="*80)
print("FedSAM Key Points:")
print("1. SAM performs two forward-backward passes per batch (2x computation)")
print("2. First pass finds adversarial perturbation (worst-case direction)")
print("3. Second pass computes gradients at perturbed point")
print("4. Encourages finding flatter minima → better generalization")
print("5. Particularly effective under heterogeneous data")
print("\nExpected result: FedSAM should achieve 3-7% higher accuracy")
print("than FedAvg in non-IID scenarios, at the cost of doubled computation")